# 🛠️ Workshop: MLOps na prática — o ciclo de vida completo de um modelo

Neste laboratório vamos montar, **em miniatura e dentro do Colab**, o ciclo de vida de um modelo
(inspirado no repositório [full-stack-on-prem-cv-mlops](https://github.com/jomariya23156/full-stack-on-prem-cv-mlops)):

```
 Dados ──► Validação ──► Treino ──► Avaliação ──► Registro ──► Deploy (API) ──► Monitoramento
   ▲         (gate)     (MLflow)    (quality gate)  (Registry)    (FastAPI)         (drift)
   └────────────────────────── retreino automático (Prefect) ◄──────────────────────┘
```

| Etapa | No repo de referência | Aqui no Colab |
|---|---|---|
| Versionamento de dados | DVC | hash do dataset (`data_version`) |
| Validação de dados | DeepChecks | checagens simples em Python |
| Rastreamento de experimentos | MLflow | MLflow (SQLite local) |
| Registro de modelos | central storage + MLflow | MLflow Model Registry (alias `champion`) |
| Orquestração | Prefect | Prefect com servidor local temporário |
| Serviço do modelo | FastAPI + Gunicorn + Nginx | FastAPI + Uvicorn |
| Log de predições | PostgreSQL | SQLite |
| Detecção de drift | Evidently | teste KS (`scipy`) |

**Como usar no Colab:** selecione um ambiente **Python 3 / CPU** e execute as células em ordem
(`Ambiente de execução → Executar tudo`). Não é necessário GPU, token ou acesso ao Drive.
A instalação e a primeira inicialização do Prefect podem levar alguns minutos; aguarde sua conclusão.

Cada execução do setup cria uma pasta própria para bancos e modelos e encerra os serviços da execução anterior.
Os arquivos são temporários: baixar o notebook não inclui esses bancos e artefatos.

## 0. Setup
Instale as dependências **antes dos imports**. O comando usa o Python do próprio kernel e interrompe a célula
se a instalação falhar. Se já tiver importado outras versões, reinicie a sessão e execute desde o início.

In [ ]:
import subprocess
import sys

if sys.version_info < (3, 11):
    raise RuntimeError("Este laboratório requer Python 3.11 ou superior; no Colab, use um runtime atualizado.")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "mlflow==3.17.2", "prefect==3.9.0",
    "numpy>=1.26,<3", "pandas>=2.2,<4", "scikit-learn>=1.6,<2",
    "scipy>=1.13,<2", "matplotlib>=3.8,<4", "requests>=2.32,<3",
    "fastapi>=0.115,<1", "uvicorn>=0.30,<1", "pydantic>=2,<3",
])
print("Dependências instaladas no kernel:", sys.executable)

In [ ]:
import os, hashlib, sqlite3, threading, time, logging, tempfile
from pathlib import Path
from importlib.metadata import version

os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
os.environ["PREFECT_SERVER_ANALYTICS_ENABLED"] = "false"
os.environ["PREFECT_SERVER_EPHEMERAL_STARTUP_TIMEOUT_SECONDS"] = "120"
os.environ["DO_NOT_TRACK"] = "1"

def encerrar_servicos():
    # Também permite executar o setup novamente sem deixar a porta 8000 ocupada.
    if "servidor" in globals():
        servidor.should_exit = True
        if "api_thread" in globals():
            api_thread.join(timeout=10)
            if api_thread.is_alive():
                raise RuntimeError("A API anterior não encerrou. Reinicie a sessão antes de continuar.")
    if "mlflow_ui" in globals() and mlflow_ui.poll() is None:
        mlflow_ui.terminate()
        mlflow_ui.wait(timeout=15)

encerrar_servicos()
WORKDIR = Path(tempfile.mkdtemp(prefix="workshop-mlops-"))
# Prefect usa seu próprio SQLite local, separado do banco do MLflow.
os.environ.setdefault("PREFECT_HOME", str(WORKDIR / "prefect"))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

logging.getLogger("mlflow").setLevel(logging.WARNING)
TRACKING_URI = f"sqlite:///{WORKDIR / 'mlflow.db'}"
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_registry_uri(TRACKING_URI)
EXPERIMENT_ID = mlflow.create_experiment(
    "workshop-digits", artifact_location=(WORKDIR / "artifacts").as_uri()
)
mlflow.set_experiment(experiment_id=EXPERIMENT_ID)
client = MlflowClient()
MODEL_NAME = "digits-classifier"
print("Python", sys.version.split()[0], "| MLflow", mlflow.__version__, "| Prefect", version("prefect"))
print("Bancos e modelos desta execução:", WORKDIR)

## 1. Dados: versionar e validar
Modelo bom começa com dado confiável. Usaremos o dataset **digits** (imagens 8×8 de dígitos manuscritos,
pixels de 0 a 16), disponível no scikit-learn sem download adicional.

- **Versionamento:** um hash identifica o conteúdo dos dados. Aqui não armazenamos histórico e metadados como o DVC.
- **Validação:** se o dado vier quebrado, o pipeline para antes de treinar.
- **Separação:** 60% para treino, 20% para validação e 20% para teste. O quality gate usa a validação;
  o teste fica reservado para simular o tráfego e medir o resultado final.

In [ ]:
X, y = load_digits(return_X_y=True)
X_train, X_rest, y_train, y_rest = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_rest, y_rest, test_size=0.5, random_state=42, stratify=y_rest
)

def versao_dados(X, y):
    h = hashlib.sha256()
    for array in (np.asarray(X), np.asarray(y)):
        h.update(str((array.shape, array.dtype.str)).encode())
        h.update(np.ascontiguousarray(array).tobytes())
    return h.hexdigest()[:10]

def validar_dados(X, y):
    X, y = np.asarray(X), np.asarray(y)
    if X.ndim != 2 or X.shape[1] != 64 or len(X) == 0:
        raise ValueError("Validação falhou: esperava uma matriz não vazia com 64 features")
    if y.ndim != 1 or len(y) != len(X):
        raise ValueError("Validação falhou: cada imagem precisa de exatamente um rótulo")
    erros = []
    if not np.issubdtype(X.dtype, np.number) or not np.isfinite(X).all():
        erros.append("pixels devem ser numéricos e finitos (sem NaN/inf)")
    elif X.min() < 0 or X.max() > 16:
        erros.append("pixels fora do intervalo [0, 16]")
    if not np.issubdtype(y.dtype, np.integer) or not np.isin(y, np.arange(10)).all():
        erros.append("rótulos devem ser inteiros de 0 a 9")
    else:
        contagens = np.bincount(y, minlength=10)
        if contagens.min() < 0.5 * contagens.mean():
            erros.append("classes ausentes ou muito desbalanceadas")
    if erros:
        raise ValueError("Validação falhou: " + "; ".join(erros))
    return True

validar_dados(X_train, y_train)
print("Treino:", X_train.shape, "| Validação:", X_val.shape, "| Teste:", X_test.shape)
print("data_version =", versao_dados(X_train, y_train))

fig, axes = plt.subplots(1, 8, figsize=(10, 1.6))
for ax, img, label in zip(axes, X_train, y_train):
    ax.imshow(img.reshape(8, 8), cmap="gray_r"); ax.set_title(str(label)); ax.axis("off")
plt.show()

In [ ]:
# Teste: o que acontece com um dado "quebrado"?
X_ruim = X_train.copy(); X_ruim[0, 0] = np.nan
try:
    validar_dados(X_ruim, y_train)
except ValueError as e:
    print(e)

## 2. Treino rastreado com MLflow
Cada treino vira um **run**: parâmetros, métricas, versão dos dados e o próprio modelo ficam registrados.
Acabou o "qual era mesmo o notebook que deu 97%?".

In [ ]:
MODELOS = {
    "logreg":        lambda p: LogisticRegression(max_iter=2000, C=p.get("C", 1.0)),
    "knn":           lambda p: KNeighborsClassifier(n_neighbors=p.get("k", 5)),
    "svm":           lambda p: SVC(C=p.get("C", 1.0), gamma="scale"),
}

def treinar(tipo, params, X_tr, y_tr, X_ev, y_ev, tags=None):
    with mlflow.start_run(run_name=tipo) as run:
        modelo = MODELOS[tipo](params).fit(X_tr, y_tr)
        pred = modelo.predict(X_ev)
        mlflow.log_params({"tipo": tipo, **params})
        mlflow.set_tags({"data_version": versao_dados(X_tr, y_tr), **(tags or {})})
        mlflow.log_metrics({"accuracy": accuracy_score(y_ev, pred),
                            "f1_macro": f1_score(y_ev, pred, average="macro")})
        modelo_logado = mlflow.sklearn.log_model(
            modelo, name="model", input_example=X_tr[:2],
            pip_requirements=[f"{p}=={version(p)}" for p in
                              ("scikit-learn", "numpy", "scipy", "cloudpickle")],
        )
        mlflow.set_tag("model_uri", modelo_logado.model_uri)
        return run.info.run_id

def uri_modelo_run(run_id):
    return client.get_run(run_id).data.tags["model_uri"]

for tipo, params in [("logreg", {"C": 0.01}), ("knn", {"k": 15}), ("svm", {"C": 5.0})]:
    treinar(tipo, params, X_train, y_train, X_val, y_val)

runs = mlflow.search_runs(experiment_ids=[EXPERIMENT_ID], order_by=["metrics.accuracy DESC"])
runs[["tags.mlflow.runName", "params.tipo", "metrics.accuracy", "metrics.f1_macro", "tags.data_version"]]

## 3. Model Registry: qual modelo está "em produção"?
Registramos o melhor run e damos a ele o **alias `champion`**. O serviço de produção nunca aponta para um arquivo solto:
ele pede `models:/digits-classifier@champion`. Trocar o modelo em produção = mover o alias.

In [ ]:
melhor_run = runs.iloc[0].run_id
mv = mlflow.register_model(uri_modelo_run(melhor_run), MODEL_NAME)
client.set_registered_model_alias(MODEL_NAME, "champion", mv.version)
print(f" {MODEL_NAME} v{mv.version} agora é o champion (run {melhor_run[:8]})")

## 4. Pipeline automatizado com Prefect
Agora empacotamos o ciclo em um **flow** com *tasks*:

`validar → treinar → quality gate → promover`

O quality gate compara o candidato com o champion **no mesmo conjunto de validação**.
Só promove se o candidato atingir a acurácia mínima e superar o champion.
Nesta demonstração desativamos o cache de tasks, porque treino, registro e promoção têm efeitos externos.
O primeiro flow inicia um servidor temporário do Prefect; não é necessário criar conta.

In [ ]:
from prefect import flow, task, get_run_logger
from prefect.cache_policies import NO_CACHE
from mlflow.exceptions import MlflowException

@task(cache_policy=NO_CACHE, persist_result=False)
def t_validar(X, y):
    return validar_dados(X, y)

@task(cache_policy=NO_CACHE, persist_result=False)
def t_treinar(X_tr, y_tr, X_ev, y_ev, motivo):
    return treinar("svm", {"C": 5.0}, X_tr, y_tr, X_ev, y_ev, tags={"motivo": motivo})

@task(cache_policy=NO_CACHE, persist_result=False)
def t_quality_gate(run_id, X_ev, y_ev, minimo=0.90):
    log = get_run_logger()
    candidato = mlflow.sklearn.load_model(uri_modelo_run(run_id))
    acc_cand = accuracy_score(y_ev, candidato.predict(X_ev))
    try:
        champion = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")
        acc_champ = accuracy_score(y_ev, champion.predict(X_ev))
    except MlflowException as exc:
        if exc.error_code != "RESOURCE_DOES_NOT_EXIST":
            raise  # Falhas no Registry não devem ser tratadas como ausência de champion.
        acc_champ = 0.0
    log.info(f"candidato={acc_cand:.3f} | champion={acc_champ:.3f} | mínimo={minimo}")
    return acc_cand >= minimo and acc_cand > acc_champ

@task(cache_policy=NO_CACHE, persist_result=False)
def t_promover(run_id):
    mv = mlflow.register_model(uri_modelo_run(run_id), MODEL_NAME)
    client.set_registered_model_alias(MODEL_NAME, "champion", mv.version)
    return mv.version

@flow(name="pipeline-treino")
def pipeline_treino(X_tr, y_tr, X_ev, y_ev, motivo="manual"):
    log = get_run_logger()
    t_validar(X_tr, y_tr)
    run_id = t_treinar(X_tr, y_tr, X_ev, y_ev, motivo)
    if t_quality_gate(run_id, X_ev, y_ev):
        versao = t_promover(run_id)
        log.info(f"🚀 Promovido: v{versao}")
        return versao
    log.warning("⛔ Reprovado no quality gate — champion mantido")
    return None

# Mesmo dado, mesmo modelo: o candidato NÃO é melhor que o champion ⇒ deve ser reprovado.
resultado_gate = pipeline_treino(X_train, y_train, X_val, y_val, motivo="teste-gate")

## 5. Deploy: o modelo vira uma API
Subimos um serviço FastAPI local que:
- carrega uma versão do `champion` do Registry;
- responde `POST /predict`, aceitando exatamente 64 pixels finitos no intervalo [0, 16];
- grava cada predição em SQLite, junto com a versão usada;
- expõe `PUT /reload` para carregar o novo champion sem encerrar o serviço.

As chamadas abaixo acontecem dentro da VM do notebook, em `127.0.0.1`.
A célula espera a API ficar pronta e permite reexecução sem duplicar o servidor.

In [ ]:
import json, requests, uvicorn
from typing import Annotated
from fastapi import FastAPI
from pydantic import BaseModel, ConfigDict, Field

if "api_thread" in globals() and api_thread.is_alive():
    servidor.should_exit = True
    api_thread.join(timeout=10)
    if api_thread.is_alive():
        raise RuntimeError("A API anterior não encerrou; reinicie a sessão.")

DB = WORKDIR / "predicoes.db"
with sqlite3.connect(DB) as con:
    con.execute("CREATE TABLE IF NOT EXISTS predicoes (ts REAL, versao TEXT, entrada TEXT, predicao INT)")

app = FastAPI(title="Digits service")
estado = {}

def carregar_champion():
    global estado
    mv = client.get_model_version_by_alias(MODEL_NAME, "champion")
    modelo = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/{mv.version}")
    # Substituição única: cada predição captura o modelo e a versão correspondentes.
    estado = {"modelo": modelo, "versao": str(mv.version)}

carregar_champion()
Pixel = Annotated[float, Field(ge=0, le=16, allow_inf_nan=False)]

class Entrada(BaseModel):
    model_config = ConfigDict(extra="forbid")
    pixels: list[Pixel] = Field(min_length=64, max_length=64)

@app.get("/health")
def health():
    return {"status": "ok", "modelo": MODEL_NAME, "versao": estado["versao"]}

@app.post("/predict")
def predict(e: Entrada):
    atual = estado
    pred = int(atual["modelo"].predict([e.pixels])[0])
    with sqlite3.connect(DB, timeout=10) as con:
        con.execute("INSERT INTO predicoes VALUES (?,?,?,?)",
                    (time.time(), atual["versao"], json.dumps(e.pixels), pred))
    return {"predicao": pred, "versao_modelo": atual["versao"]}

@app.put("/reload")
def reload():
    carregar_champion()
    return {"status": "recarregado", "versao": estado["versao"]}

API = "http://127.0.0.1:8000"

def api_json(metodo, caminho, **kwargs):
    resposta = requests.request(metodo, f"{API}{caminho}", timeout=10, **kwargs)
    resposta.raise_for_status()
    return resposta.json()

servidor = uvicorn.Server(uvicorn.Config(
    app, host="127.0.0.1", port=8000, log_level="warning", access_log=False, loop="asyncio"
))
api_thread = threading.Thread(target=servidor.run, daemon=True, name="digits-api")
api_thread.start()
prazo = time.monotonic() + 30
while time.monotonic() < prazo:
    if not api_thread.is_alive():
        raise RuntimeError("A API não iniciou. Verifique se a porta 8000 está ocupada.")
    if servidor.started:
        try:
            saude = api_json("GET", "/health")
            break
        except requests.RequestException:
            pass
    time.sleep(0.2)
else:
    servidor.should_exit = True
    api_thread.join(timeout=10)
    raise TimeoutError("A API não ficou pronta em 30 segundos.")
print(saude)

In [ ]:
# Uma chamada real à API — exatamente como um app/cliente faria.
amostra = X_test[0]
print("Esperado:", y_test[0], "| Resposta:", api_json("POST", "/predict", json={"pixels": amostra.tolist()}))
resposta_invalida = requests.post(f"{API}/predict", json={"pixels": [0.0] * 63}, timeout=10)
assert resposta_invalida.status_code == 422
print("Entrada com 63 pixels rejeitada: HTTP", resposta_invalida.status_code)

## 6. Produção muda: simulando *data drift*
O modelo foi treinado com imagens limpas. Imagine que trocaram o scanner/câmera: as imagens agora chegam
**mais claras/lavadas e com ruído**. Ninguém avisou o time de ML. O modelo continua respondendo, mas pode errar mais.

Os rótulos do conjunto de teste permitem medir o efeito nesta simulação; o detector de drift não usa esses rótulos.

In [ ]:
def camera_nova(X, seed=42):
    # imagens mais claras/lavadas + ruído: o mundo real mudou
    return np.clip(X * 0.5 + 6 + np.random.default_rng(seed).normal(0, 2.5, X.shape), 0, 16)

X_prod_ok, y_prod_ok = X_test[:150], y_test[:150]
X_prod_drift, y_prod_drift = camera_nova(X_test[150:]), y_test[150:]

def enviar_trafego(X_lote):
    return [api_json("POST", "/predict", json={"pixels": x.tolist()})["predicao"] for x in X_lote]

pred_ok = enviar_trafego(X_prod_ok)
pred_drift = enviar_trafego(X_prod_drift)
print(f"Acurácia antes da câmera nova:  {accuracy_score(y_prod_ok, pred_ok):.1%}")
print(f"Acurácia depois da câmera nova: {accuracy_score(y_prod_drift, pred_drift):.1%}  ⚠️")

fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for i in range(8):
    axes[0, i].imshow(X_test[150 + i].reshape(8, 8), cmap="gray_r"); axes[0, i].axis("off")
    axes[1, i].imshow(X_prod_drift[i].reshape(8, 8), cmap="gray_r"); axes[1, i].axis("off")
axes[0, 0].set_title("antes", loc="left"); axes[1, 0].set_title("câmera nova", loc="left")
plt.show()

### Monitoramento: detectando drift sem precisar dos rótulos
Comparamos as distribuições dos pixels recentes com as do treino usando o teste de Kolmogorov-Smirnov.
Se mais de 30% das features não constantes tiverem `p < 0,05`, emitimos um alerta.

É uma regra didática: drift não comprova perda de acurácia. A escolha do teste, dos limiares e do tratamento de
múltiplas comparações precisa ser validada para cada aplicação. O Evidently oferece uma estrutura mais ampla de monitoramento.

In [ ]:
from scipy.stats import ks_2samp

def ler_producao(ultimas=100):
    if not isinstance(ultimas, int) or ultimas <= 0:
        raise ValueError("O tamanho da janela deve ser um inteiro positivo")
    with sqlite3.connect(DB) as con:
        df = pd.read_sql_query(
            "SELECT entrada FROM predicoes ORDER BY ts DESC, rowid DESC LIMIT ?", con, params=(ultimas,)
        )
    if df.empty:
        raise ValueError("Ainda não há predições no banco; envie tráfego antes de monitorar")
    return np.array([json.loads(e) for e in df.entrada])

def checar_drift(X_ref, X_atual, alpha=0.05, limite=0.3):
    X_ref, X_atual = np.asarray(X_ref), np.asarray(X_atual)
    if (X_ref.ndim != 2 or X_atual.ndim != 2 or len(X_ref) < 2 or len(X_atual) < 2
            or X_ref.shape[1] != X_atual.shape[1]):
        raise ValueError("Drift requer duas matrizes com ao menos 2 amostras e o mesmo número de features")
    if not np.isfinite(X_ref).all() or not np.isfinite(X_atual).all():
        raise ValueError("Drift requer valores finitos")
    features = np.flatnonzero(X_ref.std(axis=0) > 0)
    if not len(features):
        raise ValueError("Não há features variáveis na referência")
    pvalores = np.array([ks_2samp(X_ref[:, j], X_atual[:, j]).pvalue for j in features])
    frac = float((pvalores < alpha).mean())
    return frac > limite, frac

for nome, janela in [("antes", X_prod_ok[:100]), ("câmera nova", ler_producao(100))]:
    drift, frac = checar_drift(X_train, janela)
    print(f"Janela '{nome}': drift={drift} ({frac:.0%} das features mudaram de distribuição)")

## 7. Fechando o ciclo: monitoramento ➜ retreino ➜ deploy automático
O flow de monitoramento:
1. lê as últimas predições do banco;
2. se detectar drift, junta dados rotulados do novo cenário ao treino;
3. dispara o pipeline com quality gate sobre a validação antiga + nova;
4. se houver novo champion, chama `PUT /reload` na API.

Para simular feedback rotulado, aplicamos a transformação da câmera **separadamente ao treino e à validação**.
Nenhuma imagem do teste entra no treino ou no quality gate. Em produção, os novos rótulos viriam de feedback ou rotulação.

Um agendamento como `.serve(name="monitor-digits", cron="0 * * * *")` exige um processo continuamente ativo;
aqui chamamos o flow manualmente na sessão do Colab.

In [ ]:
# Feedback rotulado simulado, preservando a separação das imagens de treino, validação e teste.
X_novo_tr, y_novo_tr = camera_nova(X_train, seed=43), y_train.copy()
X_novo_ev, y_novo_ev = camera_nova(X_val, seed=44), y_val.copy()

@flow(name="monitorar-e-retreinar")
def monitorar_e_retreinar():
    log = get_run_logger()
    drift, frac = checar_drift(X_train, ler_producao(100))
    log.info(f"Drift={drift} ({frac:.0%} das features)")
    if not drift:
        return "sem drift — nada a fazer"
    X_tr = np.vstack([X_train, X_novo_tr]); y_tr = np.concatenate([y_train, y_novo_tr])
    X_ev = np.vstack([X_val, X_novo_ev]); y_ev = np.concatenate([y_val, y_novo_ev])
    versao = pipeline_treino(X_tr, y_tr, X_ev, y_ev, motivo="drift-detectado")
    if versao:
        log.info(f"Deploy: {api_json('PUT', '/reload')}")
    return versao

versao_retreino = monitorar_e_retreinar()

In [ ]:
# Medida final: o mesmo tráfego de teste, ausente do treino e da seleção de modelos.
print(api_json("GET", "/health"))
pred_drift_v2 = enviar_trafego(X_prod_drift)
acc_antes = accuracy_score(y_prod_drift, pred_drift)
acc_depois = accuracy_score(y_prod_drift, pred_drift_v2)
print(f"Acurácia na câmera nova — antes: {acc_antes:.1%} | depois: {acc_depois:.1%}")
if versao_retreino:
    print("Novo champion carregado pela API:", versao_retreino)
else:
    print("O quality gate manteve o champion anterior.")

## 8. Auditoria: quem está em produção e por quê?
Tudo ficou rastreado: cada versão aponta para o run, que aponta para parâmetros, métricas, dados e o motivo do treino.

In [ ]:
versoes = client.search_model_versions(f"name='{MODEL_NAME}'")
champion = client.get_model_version_by_alias(MODEL_NAME, "champion").version
pd.DataFrame([{
    "versao": v.version,
    "champion": "⭐" if v.version == champion else "",
    "motivo": client.get_run(v.run_id).data.tags.get("motivo", "manual"),
    "data_version": client.get_run(v.run_id).data.tags.get("data_version"),
    "accuracy (avaliação)": round(client.get_run(v.run_id).data.metrics["accuracy"], 3),
} for v in sorted(versoes, key=lambda v: int(v.version))])

### (Opcional) Abrir a interface do MLflow
A célula inicia a UI na mesma base desta execução. No Colab, abra o link gerado para explorar os experimentos
durante o laboratório. O acesso pela janela do Colab depende do proxy do Google e deve ser conferido no navegador.

In [ ]:
# Reutiliza o processo se esta célula for executada novamente.
# O wildcard de hosts permite o proxy do Colab; o processo escuta somente em localhost.
import socket

if "mlflow_ui" not in globals() or mlflow_ui.poll() is not None:
    # Escolhe uma porta livre para não interferir em outros serviços locais.
    with socket.socket() as sock:
        sock.bind(("127.0.0.1", 0))
        MLFLOW_UI_PORT = sock.getsockname()[1]
    ui_log = WORKDIR / "mlflow_ui.log"
    with ui_log.open("w") as log:
        mlflow_ui = subprocess.Popen([
            sys.executable, "-m", "mlflow", "server",
            "--backend-store-uri", TRACKING_URI,
            "--host", "127.0.0.1", "--port", str(MLFLOW_UI_PORT),
            "--workers", "1", "--allowed-hosts", "*",
        ], stdout=log, stderr=subprocess.STDOUT)

ui_url = f"http://127.0.0.1:{MLFLOW_UI_PORT}"
prazo = time.monotonic() + 60
while time.monotonic() < prazo:
    if mlflow_ui.poll() is not None:
        raise RuntimeError(f"A UI não iniciou. Confira o log em {WORKDIR / 'mlflow_ui.log'}")
    try:
        if requests.get(f"{ui_url}/health", timeout=2).ok:
            break
    except requests.RequestException:
        pass
    time.sleep(0.5)
else:
    mlflow_ui.terminate()
    mlflow_ui.wait(timeout=15)
    raise TimeoutError(f"A UI não ficou pronta. Confira {WORKDIR / 'mlflow_ui.log'}")

try:
    from google.colab import output
except ImportError:
    print("Fora do Colab: abra", ui_url)
else:
    output.serve_kernel_port_as_window(MLFLOW_UI_PORT, path="/")

## 🧩 Desafios (para depois do workshop)
1. Troque o `minimo` do quality gate para `0.99`. O que acontece no retreino?
2. Deixe a `camera_nova` mais suave (ex.: `X * 0.9 + 1`). O monitor ainda detecta drift?
3. Adicione ao `t_treinar` uma busca entre `logreg`, `knn` e `svm` e promova o melhor.
4. Agende o `monitorar_e_retreinar` com `.serve(name="monitor-digits", cron=...)` e veja os runs na UI do Prefect.
5. Leia o `docker-compose.yml` do [repo de referência](https://github.com/jomariya23156/full-stack-on-prem-cv-mlops) e mapeie cada serviço para uma etapa deste notebook.

**Ao terminar:** execute `encerrar_servicos()` em uma nova célula para parar a API e a UI do MLflow.
Para refazer a demonstração, execute desde o setup; uma nova pasta será criada sem apagar os resultados anteriores.